# 🎬 Ai-mation — توليد فيديو مجاني على Google Colab (من الهاتف)

يعمل بنموذج **LTX-Video** المفتوح على كرت شاشة **T4 المجاني** من Google.

## الخطوات
1. من القائمة: **Runtime ← Change runtime type ← T4 GPU ← Save**
   (على الهاتف: اضغط ⋮ أو اسم الملف ← Runtime)
2. شغّل الخلايا **بالترتيب** بالضغط على ▶️ بجانب كل خلية.
3. الخلية 1 و2 تأخذ 5-10 دقائق في كل جلسة جديدة (تحميل النموذج ~25GB على سيرفر Google، وليس على هاتفك). بعدها كل فيديو يأخذ دقائق قليلة.
4. غيّر الوصف في الخلية 3 واضغط ▶️ لتوليد فيديو جديد — بدون إعادة الخلايا 1 و2.

> ⚠️ الجلسة المجانية محدودة بساعات يومياً، وقد لا يتوفر T4 أوقات الذروة. إذا أُغلقت الجلسة أعد تشغيل الخلايا من البداية.

In [ ]:
#@title 1️⃣ تجهيز البيئة (مرة واحدة)
import subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ لا يوجد GPU. اذهب إلى Runtime ← Change runtime type ← اختر T4 GPU ثم أعد التشغيل.")
print("✅ GPU:", gpu)
!pip install -q diffusers==0.41.0 "bitsandbytes>=0.48.0" accelerate sentencepiece imageio imageio-ffmpeg
print("✅ تم التثبيت")

In [ ]:
#@title 2️⃣ تحميل النموذج (مرة واحدة لكل جلسة)
#@markdown إذا خرج الفيديو أسود أو مشوّه، غيّر الدقة إلى bfloat16 وأعد تشغيل هذه الخلية.
precision = "float16" #@param ["float16", "bfloat16"]

import torch, gc
from transformers import T5EncoderModel, BitsAndBytesConfig
from diffusers import LTXPipeline, LTXImageToVideoPipeline

MODEL = "Lightricks/LTX-Video"
dtype = torch.float16 if precision == "float16" else torch.bfloat16

# مُرمّز النص كبير جداً، نحمله بدقة 8-bit ليتسع مع النموذج في ذاكرة T4
text_encoder = T5EncoderModel.from_pretrained(
    MODEL, subfolder="text_encoder",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map={"": 0},
)
pipe = LTXPipeline.from_pretrained(MODEL, text_encoder=text_encoder, torch_dtype=dtype)
pipe.to("cuda")
pipe.vae.enable_tiling()
i2v = LTXImageToVideoPipeline.from_pipe(pipe)
gc.collect(); torch.cuda.empty_cache()
print(f"✅ النموذج جاهز — الذاكرة المستخدمة: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
#@title 🛠️ أدوات (شغّلها مرة واحدة)
import os, re, time, random, urllib.parse, urllib.request, json
from diffusers.utils import export_to_video
from IPython.display import HTML, display
from base64 import b64encode

os.makedirs("/content/videos", exist_ok=True)
SIZES = {"طولي 9:16 (تيك توك/ريلز)": (704, 480), "عرضي 16:9 (يوتيوب)": (480, 704), "مربع 1:1": (512, 512)}
NEGATIVE = "worst quality, inconsistent motion, blurry, jittery, distorted, deformed, watermark, text, nsfw, nudity"
BLOCKED = ["nude", "naked", "nsfw", "porn", "sex", "erotic", "hentai", "topless",
           "عاري", "عارية", "جنس", "جنسي", "إباحي", "اباحي", "سكس", "بدون ملابس"]

def translate(text):
    if not re.search(r"[\u0600-\u06FF]", text):
        return text
    try:
        url = "https://api.mymemory.translated.net/get?langpair=ar|en&q=" + urllib.parse.quote(text[:500])
        t = json.load(urllib.request.urlopen(url, timeout=20))["responseData"]["translatedText"]
        print("🌐 الترجمة:", t)
        return t
    except Exception as e:
        print("⚠️ تعذرت الترجمة، سيُستخدم النص كما هو:", e)
        return text

def check(text):
    low = " " + re.sub(r"[^\w]+", " ", text.lower()) + " "
    if any(f" {w} " in low for w in BLOCKED):
        raise SystemExit("❌ الوصف يحتوي محتوى جنسياً صريحاً، وهذا غير مدعوم.")

def frames_for(seconds):
    n = int(seconds * 24)
    return (n // 8) * 8 + 1  # النموذج يحتاج عدد إطارات = 8k+1

def show(path):
    data = b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="100%" controls autoplay loop playsinline src="data:video/mp4;base64,{data}"></video>'))
    print("💾 محفوظ في:", path, "— لتحميله: افتح أيقونة 📁 على اليسار، ثم ⋮ بجانب الملف ← Download")

def run(pipeline, prompt, size, seconds, steps, seed, **extra):
    check(prompt)
    prompt_en = translate(prompt)
    check(prompt_en)
    h, w = SIZES[size]
    seed = seed if seed >= 0 else random.randint(0, 2**31 - 1)
    print(f"🎬 جارٍ التوليد... ({seconds} ثوانٍ، {steps} خطوة، seed={seed})")
    t0 = time.time()
    frames = pipeline(
        prompt=prompt_en, negative_prompt=NEGATIVE,
        width=w, height=h, num_frames=frames_for(seconds), frame_rate=24,
        num_inference_steps=steps, guidance_scale=3.0,
        generator=torch.Generator("cuda").manual_seed(seed), **extra,
    ).frames[0]
    path = f"/content/videos/aimation_{int(time.time())}.mp4"
    export_to_video(frames, path, fps=24)
    print(f"✅ تم في {time.time() - t0:.0f} ثانية")
    gc.collect(); torch.cuda.empty_cache()
    show(path)
    return path

print("✅ جاهز")

In [ ]:
#@title 3️⃣ نص ← فيديو
#@markdown اكتب الوصف بالعربي أو الإنجليزي (الوصف المفصل يعطي نتائج أفضل: الشخصية، المكان، الحركة، الإضاءة، حركة الكاميرا).
prompt = "محارب صحراوي يمشي ببطء أمام قلعة قديمة عند الغروب، الرياح تحرك عباءته، الكاميرا تتبعه بهدوء، إضاءة سينمائية" #@param {type:"string"}
size = "طولي 9:16 (تيك توك/ريلز)" #@param ["طولي 9:16 (تيك توك/ريلز)", "عرضي 16:9 (يوتيوب)", "مربع 1:1"]
seconds = 4 #@param {type:"slider", min:2, max:6, step:1}
steps = 30 #@param {type:"slider", min:15, max:50, step:5}
seed = -1 #@param {type:"integer"}
#@markdown seed = -1 يعني عشوائي. استخدم نفس الرقم لتكرار نفس النتيجة.

video_path = run(pipe, prompt, size, seconds, steps, seed)

In [ ]:
#@title 4️⃣ صورة ← فيديو (تحريك صورة)
#@markdown عند التشغيل سيظهر زر **Choose Files** لرفع صورة من هاتفك. ثم اكتب وصف الحركة.
motion = "الشخصية تبتسم وتلتفت ببطء نحو الكاميرا، الشعر يتحرك مع النسيم" #@param {type:"string"}
size = "طولي 9:16 (تيك توك/ريلز)" #@param ["طولي 9:16 (تيك توك/ريلز)", "عرضي 16:9 (يوتيوب)", "مربع 1:1"]
seconds = 4 #@param {type:"slider", min:2, max:6, step:1}
steps = 30 #@param {type:"slider", min:15, max:50, step:5}
seed = -1 #@param {type:"integer"}

from google.colab import files
from PIL import Image, ImageOps
uploaded = files.upload()
if not uploaded:
    raise SystemExit("لم يتم رفع صورة.")
name = next(iter(uploaded))
h, w = SIZES[size]
image = ImageOps.fit(Image.open(name).convert("RGB"), (w, h))
video_path = run(i2v, motion, size, seconds, steps, seed, image=image)

In [ ]:
#@title 5️⃣ (اختياري) حفظ كل الفيديوهات في Google Drive
from google.colab import drive
import shutil
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/Ai-mation"
os.makedirs(dest, exist_ok=True)
for f in os.listdir("/content/videos"):
    shutil.copy(f"/content/videos/{f}", dest)
print("✅ تم الحفظ في Google Drive ← مجلد Ai-mation")